In [1]:
!pip install networkx python-louvain


In [3]:
!wget https://snap.stanford.edu/data/roadNet-CA.txt.gz
!gunzip roadNet-CA.txt.gz

--2026-03-12 18:51:44--  https://snap.stanford.edu/data/roadNet-CA.txt.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17892860 (17M) [application/x-gzip]
Saving to: ‘roadNet-CA.txt.gz’

roadNet-CA.txt.gz   100%[===================>]  17.06M  29.7MB/s    in 0.6s    

2026-03-12 18:51:45 (29.7 MB/s) - ‘roadNet-CA.txt.gz’ saved [17892860/17892860]



In [4]:
import networkx as nx

def load_graph(file):
    G = nx.Graph()

    with open(file) as f:
        for line in f:
            if line.startswith('#'):
                continue

            u, v = map(int, line.split())
            G.add_edge(u, v)

    return G


print("Loading graph...")
G = load_graph("roadNet-CA.txt")

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Loading graph...
Nodes: 1965206
Edges: 2766607


In [5]:
import community.community_louvain as community_louvain

def detect_communities(G):

    partition = community_louvain.best_partition(G)

    communities = {}

    for node, cid in partition.items():
        communities.setdefault(cid, []).append(node)

    return list(communities.values())


print("Detecting communities...")
communities = detect_communities(G)

print("Total communities:", len(communities))

Detecting communities...
Total communities: 3025


In [18]:
def similarity(G, Ci, Q):

    edges_between = 0

    for u in Ci:
        for v in Q:
            if G.has_edge(u, v):
                edges_between += 1

    return edges_between / (len(Ci) + len(Q))

In [19]:
def top_k_search(G, communities, Q, k):

    results = []

    for Ci in communities:

        score = similarity(G, Ci, Q)
        results.append((score, Ci))

    results.sort(reverse=True)

    return results[:k]

In [16]:
Q = communities[0][:20]
results = top_k_search(communities, Q, 5)

for score, comm in results:
    print("Similarity:", score, "Community size:", len(comm))

Similarity: 0.003541703559412077 Community size: 5647
Similarity: 0.0 Community size: 4
Similarity: 0.0 Community size: 5170
Similarity: 0.0 Community size: 2
Similarity: 0.0 Community size: 2


In [21]:
!pip install networkx numpy

In [22]:
import numpy as np

# example unit patterns (vector representation)
unit_patterns = {
    "c1": np.array([1,0,1,0,1]),
    "c2": np.array([1,1,0,0,1]),
    "c3": np.array([0,1,1,0,0]),
    "c4": np.array([1,0,0,1,1]),
    "c5": np.array([0,0,1,1,0])
}

In [23]:
from numpy.linalg import norm

def cos_sim(a,b):
    return np.dot(a,b)/(norm(a)*norm(b))

In [24]:
class Node:
    def __init__(self, children=None, patterns=None):
        self.children = children if children else []
        self.patterns = patterns if patterns else []
        self.is_leaf = len(self.children)==0

In [25]:
leaf1 = Node(patterns=["c1","c2"])
leaf2 = Node(patterns=["c3"])
leaf3 = Node(patterns=["c4","c5"])

root = Node(children=[leaf1,leaf2,leaf3])

In [33]:
import heapq
import itertools

def retrieve_unit_patterns(root, query_patterns, threshold):

    candidate_set = []

    heap = []
    counter = itertools.count()

    heapq.heappush(heap, (0, next(counter), root))

    while heap:

        _, _, node = heapq.heappop(heap)

        if node.is_leaf:

            for pattern in node.patterns:

                score = 0

                for q in query_patterns:
                    score += cos_sim(unit_patterns[pattern], q)

                if score >= threshold:
                    candidate_set.append(pattern)

        else:

            for child in node.children:
                heapq.heappush(heap, (0, next(counter), child))

    return candidate_set

In [34]:
query_patterns = [
    np.array([1,0,1,0,1]),
    np.array([1,1,0,0,1])
]

threshold = 1.2

candidates = retrieve_unit_patterns(root, query_patterns, threshold)

print("Candidate unit patterns:", candidates)

Candidate unit patterns: ['c1', 'c2', 'c4']


Algorithm - 3 Splitting point on line segment

In [35]:
def distance_point_to_line(p, a, b):
    a = np.array(a)
    b = np.array(b)
    p = np.array(p)

    ab = b - a
    ap = p - a

    t = np.dot(ap, ab) / np.dot(ab, ab)
    t = max(0, min(1, t))

    closest = a + t * ab
    dist = np.linalg.norm(p - closest)

    return dist, closest, t

In [36]:
def intersection_interval(v, a, b, r):

    v = np.array(v)
    a = np.array(a)
    b = np.array(b)

    ab = b - a
    av = a - v

    A = np.dot(ab, ab)
    B = 2 * np.dot(av, ab)
    C = np.dot(av, av) - r*r

    disc = B*B - 4*A*C

    if disc < 0:
        return None

    t1 = (-B - np.sqrt(disc)) / (2*A)
    t2 = (-B + np.sqrt(disc)) / (2*A)

    return (min(t1, t2), max(t1, t2))

In [40]:
def find_splitting_points(L_start, L_end, unit_patterns, r):

    S = []
    intervals = []

    for pattern in unit_patterns:

        interval_list = []

        for v in pattern:

            inter = intersection_interval(v, L_start, L_end, r)

            if inter is None:
                continue

            t1, t2 = inter

            t1 = max(0, t1)
            t2 = min(1, t2)

            if t1 <= t2:
                interval_list.append((t1, t2))

        intervals.extend(interval_list)

    for t1, t2 in intervals:
      S.append(t1)
      S.append(t2)
      S = sorted(set(float(x) for x in S))
    return S

In [41]:
# line segment L
L_start = (0,0)
L_end = (10,0)

# unit patterns (list of vertex sets)
unit_patterns = [
    [(2,1),(3,2)],
    [(6,1),(7,-1)],
    [(5,3)]
]

r = 2

In [42]:
S = find_splitting_points(L_start, L_end, unit_patterns, r)

print("Splitting points:", S)

Splitting points: [0.026794919243112253, 0.3, 0.3732050807568878, 0.4267949192431122, 0.5267949192431122, 0.7732050807568878, 0.8732050807568877]


Algorithm 4: CT op-kCS2 Query Answering.

In [43]:
def ctopkcs_query(L_start, L_end, G, unit_patterns, r, k, root):

    # Step 1: find splitting points (Algorithm 3)
    S = find_splitting_points(L_start, L_end, unit_patterns, r)

    results = []

    # Step 2: process each interval on the query line
    for i in range(len(S)-1):

        interval_center = (S[i] + S[i+1]) / 2

        # Step 3: obtain query unit patterns
        query_patterns = []
        for pattern in unit_patterns:
            query_patterns.append(np.array(pattern[0]))   # simple representation

        # Step 4: retrieve candidate unit patterns (Algorithm 2)
        candidates = retrieve_unit_patterns(root, query_patterns, threshold=1.0)

        # Step 5: compute top-k communities (Algorithm 1)
        topk = top_k_search(communities, query_patterns[0], k)

        results.append(topk)

    return results